# Retrieval & generation  with llama Stack & Milvus on wx.data


#### Prerequesites

- run one foundational model on RHOAI
- run one embedding model on RHOAI
- run Llama Stack Server server in version 0.5.x on RHOAI (using remote::vllm providers) with enabled inline Milvus
- documents embeded and added to a the Milvus collection - (see Indexing notebook wwith llama stack)

Add environment variables
- **LLAMASTACK_URL**
- **LLAMASTACK_APIKEY** if auth/authz is enabled on Llama Stack Server

NOTE: You can run also LLama Stack locally, using either remote::vllm (for remote models) or ollama (for local models) providers.



In [ ]:
!pip install dotenv
!pip install llama_stack_client==0.5.0

#### Import dependencies

**NOTE:** This assumes adding environment variables from `.env` file.

In [ ]:
import logging
import os

from dotenv import load_dotenv

load_dotenv()

logging.getLogger("httpx").setLevel(logging.WARNING)

#### Create LlamaStackClient object

**NOTE:** If you do not have authorization enabled on your Llama Stack instance, just skip providing `LLAMASTACK_APIKEY`.


In [ ]:
from llama_stack_client import LlamaStackClient

client = LlamaStackClient(
    base_url=os.getenv("LLAMASTACK_URL", "http://localhost:5321"),
    api_key=os.getenv("LLAMASTACK_APIKEY"),
)

#### Models used in the example

In [3]:
EMBEDDING_MODEL = "vllm-embedding/granite-278m-multilingual-1"
EMBEDDING_MODEL_DIMENSION = 768

LLM_MODEL = "vllm-inference-llama-3-1/redhataillama-31-8b-instruct"

In [4]:
client.models.list()[0].model_dump()

{'id': 'vllm-inference-llama-3-1/RedHatAI/Llama-3.1-8B-Instruct',
 'created': 1771842950,
 'owned_by': 'llama_stack',
 'custom_metadata': {'model_type': 'llm',
  'provider_id': 'vllm-inference-llama-3-1',
  'provider_resource_id': 'RedHatAI/Llama-3.1-8B-Instruct'},
 'object': 'model'}

#### Tests model availability via llama stack

In [5]:
msgs = [{"role": "user", "content": "Generate a short poem"}]

llm_resp = client.chat.completions.create(messages=msgs, model=LLM_MODEL)

llm_resp.model_dump()

{'id': 'chatcmpl-0942a7dc0a094e43ae0ba7816c76fcb9',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'content': "Softly falls the evening dew,\nA gentle hush, a peaceful view,\nThe stars appear, a twinkling sea,\nA night of rest, for you and me.\n\nThe world is still, in quiet sleep,\nThe moon's soft light, our souls do keep,\nIn this calm moment, we find our peace,\nA sense of calm, our worries release.",
    'name': None,
    'role': 'assistant',
    'tool_calls': None,
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'reasoning': None,
    'reasoning_content': None},
   'logprobs': None,
   'stop_reason': None,
   'token_ids': None}],
 'created': 1771842951,
 'model': 'vllm-inference-llama-3-1/redhataillama-31-8b-instruct',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 75,
  'prompt_tokens': 39,
  'total_tokens': 114,
  'completion_tokens_details': None,
  'prompt_tokens_details': None},
 'service_tie

In [6]:
emb_response = client.embeddings.create(input="Hello", model=f"{EMBEDDING_MODEL}")
emb_response.model_dump()

{'data': [{'embedding': [-0.01421553734689951,
    0.02087596245110035,
    -0.00365329347550869,
    0.014414356090128422,
    0.03260626643896103,
    -0.05805505812168121,
    0.028232255950570107,
    0.043143656104803085,
    0.048312943428754807,
    0.003305360907688737,
    0.004423716105520725,
    0.014712583273649216,
    0.008151566609740257,
    -0.007903043180704117,
    0.02027950808405876,
    0.07634638249874115,
    0.07356292009353638,
    -0.05765742436051369,
    -0.030618079006671906,
    -0.002708904678002,
    -0.012078235857188702,
    0.035588547587394714,
    -0.027238162234425545,
    0.07952748239040375,
    0.020478326827287674,
    -0.05368104949593544,
    -0.013917309232056141,
    -0.027039343491196632,
    -0.06401962041854858,
    0.03598618507385254,
    0.012674692086875439,
    0.00835038535296917,
    -0.01640254259109497,
    0.050698768347501755,
    -0.0020875963382422924,
    0.004200045019388199,
    0.010736209340393543,
    0.0328050851821

### Files used as benchmarking data

**NOTE:** The below example assumes that benchmarking data resides in subfolder: `data/`. Moreover, this example itself does not deliver these artifacts - you need to replace:
- `benchmark_qa_path` with filename of benchmarking dataset, against which we'll test retrieval

In [7]:
import json
from pathlib import Path

In [8]:
docs_folder_path = Path("data/")
benchmark_qa_path = docs_folder_path.joinpath(
    "ibm_annual_report_pdf_benchmarking_data.json"
)

In [9]:
benchmark_qa = json.loads(benchmark_qa_path.read_bytes())
benchmark_qa[:3]

[{'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What was the operating other income and expense in 2024?',
  'correct_answer': '$1,656 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What is the amount of gain on land/building dispositions included in "Other"',
  'correct_answer': '$126 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'How much was the non-operating retirement-related costs in the current-year period?',
  'correct_answer': '$3,457 million'}]

### Choosing vector store with knowledge base

**NOTE:** Use `llamastack_rag_index_building.ipynb` notebook to build knowledge base, stored in vector store - reuse the same vector store ID in this example.

In [10]:
client.vector_stores.list()

SyncOpenAICursorPage[VectorStore](data=[VectorStore(id='vs_b1ac3edf-6a01-45fb-afaf-d01c5a335d53', created_at=1771842873, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), expires_after=None, expires_at=None, last_active_at=1771842873, metadata={'provider_id': 'milvus', 'provider_vector_store_id': 'vs_b1ac3edf-6a01-45fb-afaf-d01c5a335d53', 'embedding_model': 'vllm-embedding/granite-278m-multilingual-1', 'embedding_dimension': '768'}, name=None, object='vector_store', status='completed', usage_bytes=0), VectorStore(id='vs_046d8729-2f72-4ec6-ba89-c3979573c12a', created_at=1771840134, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), expires_after=None, expires_at=None, last_active_at=1771840134, metadata={'provider_id': 'milvus', 'provider_vector_store_id': 'vs_046d8729-2f72-4ec6-ba89-c3979573c12a', 'embedding_model': 'vllm-embedding/granite-278m-multilingual-1', 'embedding_dimension': '768'}, name=None, object='vector_st

In [11]:
vector_store_id = "vs_046d8729-2f72-4ec6-ba89-c3979573c12a"

### Build user prompt

In [12]:
DEFAULT_PROMPT_TEMPLATE = "{reference_documents}\n[conversation]: {question}. Answer with no more than 150 words. If you cannot base your answer on the given document, please state that you do not have an answer. Respond exclusively in the language of the question, regardless of any other language used in the provided context. Ensure that your entire response is in the same language as the question.\n"

DEFAULT_CONTEXT_TEMPLATE = "[document]: {document}\n"

DEFAULT_SYSTEM_PROMPT = "You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.\nIf a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don’t know the answer to a question, please don’t share false information.\n"

In [13]:
def build_prompt(
    question: str,
    reference_documents: list[str] = None,
    prompt_template_text: str = DEFAULT_PROMPT_TEMPLATE,
    context_template_text: str = DEFAULT_CONTEXT_TEMPLATE,
) -> str:
    """
    Warning: It's simplified prompt builder, without sampling of the reference documents
    """
    if reference_documents:
        reference_documents = [
            context_template_text.format(document=reference_document)
            for reference_document in reference_documents
        ]
    else:
        reference_documents = []
    prompt_variables = {
        "question": question,
        "reference_documents": "\n".join(reference_documents),
    }
    return prompt_template_text.format(**prompt_variables)

### Retrieval & generation

In [14]:
example_question = benchmark_qa[-1].get("question")
example_question

'What was the total revenue for year 2024?'

In [15]:
search_response = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=example_question,
    max_num_results=5,
)
search_response.model_dump()

{'data': [{'content': [{'text': 'S o f t w a r e r e v e n u e g r e w 9 % a t c o n s t a n t c u r r e n c y , r e f l e c t i n g\ns t r o n g d e m a n d f o r o u r a d v a n c e d c a p a b i l i t i e s i n h y b r i d c l o u d ,',
     'type': 'text',
     'chunk_metadata': None,
     'embedding': None,
     'metadata': None}],
   'file_id': '',
   'filename': '',
   'score': 0.6445577144622803,
   'attributes': {}},
  {'content': [{'text': '( 1 )\n2 0 2 2\n( 1 )\nY r . - t o - Y r .\nP e r c e n t /\nM a r g i n\nC h a n g e\nY r . - t o - Y r .\nP e r c e n t C h a n g e\nA d j u s t e d f o r\nC u r r e n c y\nR e v e n u e\nS o f t w a r e $ 2 5 , 0 1 1 $ 2 3 , 6 2 9 5 . 8 % 5 . 9 %',
     'type': 'text',
     'chunk_metadata': None,
     'embedding': None,
     'metadata': None}],
   'file_id': '',
   'filename': '',
   'score': 0.633152186870575,
   'attributes': {}}],
 'search_query': ['What was the total revenue for year 2024?'],
 'has_more': False,
 'next_page': None,

In [16]:
reference_documents = [doc.content[0].text for doc in search_response.data]
len(reference_documents)

2

In [17]:
user_prompt = build_prompt(
    question=example_question, reference_documents=reference_documents
)

In [18]:
msgs = [
    {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt},
]


response_chat = client.chat.completions.create(model=LLM_MODEL, messages=msgs)
answer = response_chat.choices[0].message.content
answer

'I do not have an answer. The provided document only contains information about the software revenue growth and percentage change from year 2022 to 2023, but it does not include any data for year 2024.'